# 01 Data exploration

## Kolonner
### Sjekke at nødvendige kolonner er tilstede:

In [ ]:
import pandas as pd

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

required_columns = ["MeteringPointAnonymous","TimestampUtc","Value","TransformerStation","ConsumptionCode"]  # legg til kolonner dere forventer

for f in files:
    df = pd.read_csv(f, nrows=10)  # les kun de første 10 radene
    print(f"\n{f}:")
    # sjekk om alle forventede kolonner er der
    has_all = all(col in df.columns for col in required_columns)
    print("Alle nødvendige kolonner tilstede:", "JA" if has_all else "NEI")


../data/raw/BREIVE.csv:
Alle nødvendige kolonner tilstede: JA

../data/raw/FRIKSTAD.csv:
Alle nødvendige kolonner tilstede: JA

../data/raw/HARTEVATN.csv:
Alle nødvendige kolonner tilstede: JA

../data/raw/TIMENES TS.csv:
Alle nødvendige kolonner tilstede: JA


## Tidsrom

In [6]:
for f in files:
    df_iter = pd.read_csv(f, usecols=["TimestampUtc"], parse_dates=["TimestampUtc"], chunksize=1000000)
    first, last = None, None
    for chunk in df_iter:
        if first is None:
            first = chunk["TimestampUtc"].min()
        last = chunk["TimestampUtc"].max()
    print(f"{f}: Start {first}, Slutt {last}")

../data/raw/BREIVE.csv: Start 2024-11-01 23:00:00, Slutt 2026-01-20 22:00:00
../data/raw/FRIKSTAD.csv: Start 2024-11-03 23:00:00, Slutt 2026-01-17 22:00:00
../data/raw/HARTEVATN.csv: Start 2024-11-03 23:00:00, Slutt 2026-01-17 22:00:00
../data/raw/TIMENES TS.csv: Start 2024-11-03 23:00:00, Slutt 2026-01-17 22:00:00


## Missing values

In [19]:
required_columns = ["MeteringPointAnonymous","TimestampUtc","Value","TransformerStation","ConsumptionCode"] 

for f in files:
    df_iter = pd.read_csv(f, usecols=required_columns, chunksize=1000000)
    missing = False
    for chunk in df_iter:
        if chunk.isna().any().any():
            missing = True
            break
    print(f"{f}: Mangler verdier:", "JA" if missing else "NEI")

../data/raw/BREIVE.csv: Mangler verdier: NEI
../data/raw/FRIKSTAD.csv: Mangler verdier: NEI
../data/raw/HARTEVATN.csv: Mangler verdier: NEI
../data/raw/TIMENES TS.csv: Mangler verdier: NEI


### Nullverdier
Sjekke om noen verdier er 0000...

In [26]:
import pandas as pd

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

chunksize = 1_000_000

for file in files:
    print(f"\nSjekker {file} ...")
    
    error_lines = {
        "TimestampUtc_zero": [],
        "Value_zero": [],
        "ConsumptionCode_zero": []
    }
    
    line_offset = 0
    
    for chunk in pd.read_csv(file, chunksize=chunksize):
        # 1️⃣ TimestampUtc = "0000..." eller NaN
        mask = chunk["TimestampUtc"].astype(str).str.startswith("0000") | chunk["TimestampUtc"].isna()
        error_lines["TimestampUtc_zero"].extend((mask[mask].index + line_offset).tolist())
        
        # 2️⃣ Value = 0 eller NaN
        mask = (chunk["Value"] == 0) | chunk["Value"].isna()
        error_lines["Value_zero"].extend((mask[mask].index + line_offset).tolist())
        
        # 3️⃣ ConsumptionCode = 0 eller NaN
        mask = (chunk["ConsumptionCode"] == 0) | chunk["ConsumptionCode"].isna()
        error_lines["ConsumptionCode_zero"].extend((mask[mask].index + line_offset).tolist())
        
        line_offset += len(chunk)
    
    # Skriv ut resultater per fil
    for col, lines in error_lines.items():
        if lines:
            print(f"{col} har 0 eller null på linjer: {lines[:20]}{' ...' if len(lines)>20 else ''} (totalt {len(lines)} rader)")
        else:
            print(f"{col} ingen 0-verdier")


Sjekker ../data/raw/BREIVE.csv ...
TimestampUtc_zero ingen 0-verdier
Value_zero har 0 eller null på linjer: [8007, 8011, 14136, 14137, 14138, 14139, 14140, 14141, 14142, 14143, 14144, 14145, 14146, 14147, 14148, 14149, 14150, 14151, 14152, 14153] ... (totalt 36269 rader)
ConsumptionCode_zero ingen 0-verdier

Sjekker ../data/raw/FRIKSTAD.csv ...
TimestampUtc_zero ingen 0-verdier
Value_zero har 0 eller null på linjer: [1184, 1185, 1188, 1189, 1190, 1191, 1192, 1196, 1197, 1198, 1199, 1200, 1203, 1204, 1205, 1206, 1207, 3176, 3177, 3179] ... (totalt 342116 rader)
ConsumptionCode_zero ingen 0-verdier

Sjekker ../data/raw/HARTEVATN.csv ...
TimestampUtc_zero ingen 0-verdier
Value_zero har 0 eller null på linjer: [744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763] ... (totalt 244185 rader)
ConsumptionCode_zero ingen 0-verdier

Sjekker ../data/raw/TIMENES TS.csv ...
TimestampUtc_zero ingen 0-verdier
Value_zero har 0 eller null på linjer: [2436,

## Riktig format
### Tidsformat:

In [9]:
import pandas as pd

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

for f in files:
    print(f"\nSjekker {f} ...")
    all_good = True
    # les kun tidskolonnen, i biter
    for chunk in pd.read_csv(f, usecols=["TimestampUtc"], chunksize=1000000):
        try:
            # prøv å konvertere til datetime
            pd.to_datetime(chunk["TimestampUtc"], errors="raise")
        except Exception as e:
            print("Feil i tidsformat i chunk:", e)
            all_good = False
            break
    print("Alle timestamps i samme format:", "JA" if all_good else "NEI")


Sjekker ../data/raw/BREIVE.csv ...
Alle timestamps i samme format: JA

Sjekker ../data/raw/FRIKSTAD.csv ...
Alle timestamps i samme format: JA

Sjekker ../data/raw/HARTEVATN.csv ...
Alle timestamps i samme format: JA

Sjekker ../data/raw/TIMENES TS.csv ...
Alle timestamps i samme format: JA


### Format i alle kolonner

In [23]:
import pandas as pd

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

for f in files:
    print(f"\nSjekker {f} ...")
    correct_format = True
    
    for chunk in pd.read_csv(f, chunksize=1_000_000):
        # 1️⃣ MeteringPointAnonymous → sjekk string/UUID-lignende
        if not chunk["MeteringPointAnonymous"].apply(lambda x: isinstance(x, str) and len(x)==36).all():
            print("Feil i MeteringPointAnonymous")
            correct_format = False
            break
        
        # 2️⃣ TimestampUtc → datetime
        try:
            pd.to_datetime(chunk["TimestampUtc"], errors="raise")
        except Exception as e:
            print("Feil i TimestampUtc:", e)
            correct_format = False
            break
        
        # 3️⃣ Value → float
        if not pd.to_numeric(chunk["Value"], errors="coerce").notna().all():
            print("Feil i Value")
            correct_format = False
            break
        
        # 4️⃣ TransformerStation → sjekk mot kjent liste
        valid_stations = ["BREIVE","FRIKSTAD","HARTEVATN","TIMENES TS"]
        if not chunk["TransformerStation"].isin(valid_stations).all():
            print("Feil i TransformerStation")
            correct_format = False
            break
        
        # 5️⃣ ConsumptionCode → int
        if not pd.to_numeric(chunk["ConsumptionCode"], errors="coerce").notna().all():
            print("Feil i ConsumptionCode")
            correct_format = False
            break
    
    print("Alle kolonner korrekt format:", "JA" if correct_format else "NEI")


Sjekker ../data/raw/BREIVE.csv ...
Alle kolonner korrekt format: JA

Sjekker ../data/raw/FRIKSTAD.csv ...
Alle kolonner korrekt format: JA

Sjekker ../data/raw/HARTEVATN.csv ...
Alle kolonner korrekt format: JA

Sjekker ../data/raw/TIMENES TS.csv ...
Alle kolonner korrekt format: JA


## Consumption code (type husholdning)
### Hvor mange tilfeller av hver kode:

In [25]:
import pandas as pd
from collections import Counter

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

for f in files:
    print(f"\nSjekker {f} ...")
    counter = Counter()
    
    # les kun kolonnen med ConsumptionCode, chunk for chunk
    for chunk in pd.read_csv(f, usecols=["ConsumptionCode"], chunksize=1000000):
        counter.update(chunk["ConsumptionCode"].value_counts().to_dict())
    
    print("Antall forskjellige koder:", len(counter))
    print("Oversikt (ConsumptionCode : antall):")
    for code, count in counter.items():
        print(f"{code} : {count}")


Sjekker ../data/raw/BREIVE.csv ...
Antall forskjellige koder: 3
Oversikt (ConsumptionCode : antall):
36 : 5224128
35 : 512256
26 : 101568

Sjekker ../data/raw/FRIKSTAD.csv ...
Antall forskjellige koder: 3
Oversikt (ConsumptionCode : antall):
35 : 24058368
36 : 3095616
26 : 132480

Sjekker ../data/raw/HARTEVATN.csv ...
Antall forskjellige koder: 3
Oversikt (ConsumptionCode : antall):
36 : 8319744
35 : 1307136
26 : 61824

Sjekker ../data/raw/TIMENES TS.csv ...
Antall forskjellige koder: 3
Oversikt (ConsumptionCode : antall):
35 : 13777920
36 : 1682496
26 : 1073088


### Hvor mange unike id-er av hver kode:

In [11]:
import pandas as pd
from collections import defaultdict

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

for f in files:
    print(f"\nSjekker {f} ...")
    
    # defaultdict lagrer ConsumptionCode for hver husholdning
    household_codes = defaultdict(set)
    
    # Chunk-lesing
    for chunk in pd.read_csv(f, usecols=["MeteringPointAnonymous", "ConsumptionCode"], chunksize=1000000):
        for hh, code in zip(chunk["MeteringPointAnonymous"], chunk["ConsumptionCode"]):
            household_codes[hh].add(code)
    
    # Antall unike husholdninger
    num_households = len(household_codes)
    print("Antall husholdninger:", num_households)
    
    # Oversikt over hvor mange husholdninger per ConsumptionCode
    code_counter = defaultdict(int)
    for codes in household_codes.values():
        for c in codes:
            code_counter[c] += 1
    
    print("Antall husholdninger per ConsumptionCode:")
    for code, count in code_counter.items():
        print(f"{code} : {count}")


Sjekker ../data/raw/BREIVE.csv ...
Antall husholdninger: 1322
Antall husholdninger per ConsumptionCode:
36 : 1183
35 : 116
26 : 23

Sjekker ../data/raw/FRIKSTAD.csv ...
Antall husholdninger: 6179
Antall husholdninger per ConsumptionCode:
35 : 5448
36 : 701
26 : 30

Sjekker ../data/raw/HARTEVATN.csv ...
Antall husholdninger: 2194
Antall husholdninger per ConsumptionCode:
36 : 1884
35 : 296
26 : 14

Sjekker ../data/raw/TIMENES TS.csv ...
Antall husholdninger: 3744
Antall husholdninger per ConsumptionCode:
35 : 3120
26 : 243
36 : 381


## Kode 36 i 2024?
### Sjekker om koden kan bety norgespris..

In [18]:
import pandas as pd

files = [
    "../data/raw/BREIVE.csv",
    "../data/raw/FRIKSTAD.csv",
    "../data/raw/HARTEVATN.csv",
    "../data/raw/TIMENES TS.csv"
]

for f in files:
    print(f"\nSjekker {f} ...")
    found = False
    
    # Chunk-lesing
    for chunk in pd.read_csv(f, usecols=["MeteringPointAnonymous", "ConsumptionCode", "TimestampUtc"], 
        chunksize=1_000_000, parse_dates=["TimestampUtc"]):
        # Filtrer for kode 36
        mask_code = chunk["ConsumptionCode"] == 36
        # Filtrer for år 2024
        mask_year = chunk["TimestampUtc"].dt.year == 2024
        if (mask_code & mask_year).any():
            found = True
            break  # stopper ved første funn
    
    print("Finnes husholdning med kode 36 i 2024:", "JA" if found else "NEI")


Sjekker ../data/raw/BREIVE.csv ...
Finnes husholdning med kode 36 i 2024: JA

Sjekker ../data/raw/FRIKSTAD.csv ...
Finnes husholdning med kode 36 i 2024: JA

Sjekker ../data/raw/HARTEVATN.csv ...
Finnes husholdning med kode 36 i 2024: JA

Sjekker ../data/raw/TIMENES TS.csv ...
Finnes husholdning med kode 36 i 2024: JA
